[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/11-ai-automation/notebooks/02-webhooks-fastapi-offline.ipynb)

# Notebook 02 — Webhooks with FastAPI (fully offline)

**Goal:** build a webhook receiver and test it **without running a server**, using FastAPI's `TestClient` (it calls the app directly in memory).

A **webhook** is an HTTP POST that another system sends to your URL when something happens — a doorbell instead of checking the door. We implement the four rules of reliable webhooks:

1. **verify** the sender (HMAC signature),
2. **validate** the data (Pydantic),
3. be **idempotent** (ignore duplicate deliveries),
4. **reply fast**, do slow work in the background.

## Step 1 — Signatures

An **HMAC** is a fingerprint of the message made with a shared secret. Change one character of the body, or use a different secret, and the fingerprint changes completely. The sender puts it in a header; we recompute and compare.

In [1]:
import hmac, hashlib, json

SECRET = "dev-secret-change-me"      # in real life: from an environment variable
def sign(body: bytes, secret: str = SECRET) -> str:
    return hmac.new(secret.encode(), body, hashlib.sha256).hexdigest()

body = b'{"hello": "world"}'
print(sign(body))
print(sign(b'{"hello": "World"}'))          # one letter changed -> totally different
print(sign(body, "wrong-secret"))

8ce93ae1c4ca42d984161f11e6ddfe79729f55a61ca08a8fe3396e528ecb5d5f
bed44fd9d9e1c6576a642020ce7dd4d22e1d63d2ce40e2ce41a4801e793e49eb
603d2b357449a980c98d20d192d5f0034ebff2950cdd6c30f321b5cad2db01c4


## Step 2 — The FastAPI app

* A **Pydantic** model `Ticket` describes the expected JSON. Because we read the raw body ourselves (we need the exact bytes for the signature), we validate it with `model_validate_json` and turn a `ValidationError` into HTTP **422** ("unprocessable data").
* `hmac.compare_digest` compares signatures safely (same time whether they match or not).
* `BackgroundTasks` runs `notify` **after** the reply is sent.
* `SEEN` remembers event ids, so a repeated delivery is acknowledged but not processed twice.

In [2]:
from fastapi import FastAPI, Request, Header, HTTPException, BackgroundTasks
from pydantic import BaseModel, ValidationError

app = FastAPI()
SEEN, OUTBOX = set(), []

class Ticket(BaseModel):
    event_id: str
    email: str
    subject: str
    message: str

def classify(t: Ticket) -> str:
    text = (t.subject + " " + t.message).lower()
    if any(w in text for w in ["refund", "charged", "invoice"]): return "billing"
    if any(w in text for w in ["down", "error", "urgent", "crash"]): return "urgent"
    return "general"

def notify(category: str, t: Ticket):
    OUTBOX.append(f"to {'#support-urgent' if category == 'urgent' else 'support@example.com'}: [{category}] {t.subject}")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/webhook/ticket")
async def receive(request: Request, background: BackgroundTasks, x_signature: str = Header(default="")):
    raw = await request.body()
    if not hmac.compare_digest(sign(raw), x_signature):
        raise HTTPException(401, "bad signature")
    try:
        t = Ticket.model_validate_json(raw)
    except ValidationError as e:        # bad or missing fields -> 422 Unprocessable
        raise HTTPException(422, [{"loc": err["loc"], "msg": err["msg"]} for err in e.errors()])
    if t.event_id in SEEN:
        return {"status": "duplicate"}
    SEEN.add(t.event_id)
    category = classify(t)
    background.add_task(notify, category, t)
    return {"status": "accepted", "category": category}
print("routes:", [r.path for r in app.routes][-2:])

routes: ['/health', '/webhook/ticket']


## Step 3 — Test it with `TestClient`

`TestClient` behaves like `httpx` but sends requests straight into the app. A helper builds a signed request exactly like a real sender would.

In [3]:
from fastapi.testclient import TestClient
client = TestClient(app)

def send(payload: dict, secret: str = SECRET):
    raw = json.dumps(payload).encode()
    return client.post("/webhook/ticket", content=raw,
                       headers={"Content-Type": "application/json", "X-Signature": sign(raw, secret)})

print("health:", client.get("/health").json())
ticket = {"event_id": "evt_001", "email": "asha@example.com", "subject": "Site is down", "message": "Error 500, urgent"}
r = send(ticket); print("good:     ", r.status_code, r.json())
r = send(ticket); print("duplicate:", r.status_code, r.json())
r = send({**ticket, "event_id": "evt_002"}, secret="hacker"); print("bad sig:  ", r.status_code, r.json())
raw = json.dumps({"event_id": "evt_003"}).encode()
r = client.post("/webhook/ticket", content=raw, headers={"X-Signature": sign(raw)}); print("bad data: ", r.status_code, r.json()["detail"][0]["msg"])
print("OUTBOX:", OUTBOX)

health: {'status': 'ok'}
good:      200 {'status': 'accepted', 'category': 'urgent'}
duplicate: 200 {'status': 'duplicate'}
bad sig:   401 {'detail': 'bad signature'}
bad data:  422 Field required
OUTBOX: ['to #support-urgent: [urgent] Site is down']


Note: the duplicate was accepted with `200` (so the sender stops retrying) but **not** processed again — the OUTBOX has one entry, not two.

## Step 4 — Simulate a stream of events, with retries

Real senders retry when your reply is slow or fails. Below, a sender delivers 5 events; some are delivered twice (as happens after timeouts). The receiver processes each event exactly once.

In [4]:
import random
random.seed(7)
OUTBOX.clear()
events = [
    {"event_id": f"evt_1{i}", "email": f"user{i}@example.com", "subject": s, "message": m}
    for i, (s, m) in enumerate([("Refund please", "I was charged twice"), ("Crash", "App crashes on login"),
                                ("Hello", "Do you ship to Taiwan?"), ("Invoice", "Invoice missing"), ("Outage", "Site down, urgent")])
]
deliveries = []
for e in events:
    deliveries.append(e)
    if random.random() < 0.5:
        deliveries.append(e)                 # retry -> duplicate delivery
results = [send(e).json()["status"] for e in deliveries]
print(len(deliveries), "deliveries:", results)
print(len(OUTBOX), "notifications sent (one per unique event):")
for line in OUTBOX: print("  ", line)

8 deliveries: ['accepted', 'duplicate', 'accepted', 'duplicate', 'accepted', 'accepted', 'duplicate', 'accepted']
5 notifications sent (one per unique event):
   to support@example.com: [billing] Refund please
   to #support-urgent: [urgent] Crash
   to support@example.com: [general] Hello
   to support@example.com: [billing] Invoice
   to #support-urgent: [urgent] Outage


## Step 5 — A search endpoint for the RAG workflow

n8n workflow 03 calls a `/search` endpoint to find relevant passages. Here is that endpoint (BM25 keyword ranking from module 09), tested the same way.

In [5]:
from rank_bm25 import BM25Okapi

FAQ = ["Refunds are processed within 5 working days to the original payment method.",
       "You can reset your password from the login page using 'Forgot password'.",
       "Our support team is available Monday to Friday, 9am to 6pm IST.",
       "Invoices can be downloaded from Settings > Billing > Invoices."]
bm25 = BM25Okapi([d.lower().split() for d in FAQ])

class Query(BaseModel):
    question: str
    k: int = 2

@app.post("/search")
def search(q: Query):
    scores = bm25.get_scores(q.question.lower().split())
    best = sorted(range(len(FAQ)), key=lambda i: -scores[i])[: q.k]
    return {"question": q.question, "context": [FAQ[i] for i in best if scores[i] > 0]}

print(client.post("/search", json={"question": "how long do refunds take"}).json())
print(client.post("/search", json={"question": "zebra"}).json())

{'question': 'how long do refunds take', 'context': ['Refunds are processed within 5 working days to the original payment method.']}
{'question': 'zebra', 'context': []}


The second query found nothing, so `context` is empty — exactly the case workflow 03's **IF "Found context?"** node sends to the polite "no answer" reply.

## Step 6 — Running it for real (outside the notebook)

```bash
cd 11-ai-automation/code
export WEBHOOK_SECRET=change-me
uvicorn webhook_receiver:app --port 8000      # terminal 1
python send_test_webhook.py                   # terminal 2
```

To let an outside service (or n8n Cloud) reach your laptop, use a tunnel tool such as ngrok or Cloudflare Tunnel; it gives you a temporary public URL that forwards to `localhost:8000`.

## Recap

* Verify → validate → de-duplicate → reply fast, work in background.
* `TestClient` lets you test webhooks with no server and no network.
* Return `200` for duplicates so senders stop retrying.

**Try it:** add a timestamp header `X-Timestamp` and reject requests older than 5 minutes (this blocks **replay attacks**, where an attacker re-sends an old, validly signed request).